# Extract Job Metadata from Text Fields

**Purpose:** Extract structured data (job_type, level, is_remote) from unstructured text fields (description, requirements_text, required_skills)

**Fields to extract:**
- `job_type`: Full-time, Part-time, Contract, Freelance, Internship
- `level`: Entry, Junior, Mid, Senior, Lead, Manager, Director, Executive
- `is_remote`: 0 (on-site), 1 (remote/hybrid)

**Run cells in order**

## 1. Install Dependencies

In [1]:
!pip install pymysql sqlalchemy pandas tqdm cryptography

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 2.3 MB/s eta 0:00:00


## 2. Import Libraries

In [2]:
import pymysql
import pandas as pd
import json
import re
import ssl
import tempfile
import os
from sqlalchemy import create_engine, text
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')
print('✅ Import OK')

✅ Import OK


## 3. Database Configuration

**IMPORTANT:** Update database credentials and SSL certificate

In [3]:
# ========== DATABASE CONFIG ==========
DB_CONFIG = {
    'host': 'gateway01.ap-southeast-1.prod.aws.tidbcloud.com',
    'port': 4000,
    'user': '4GJhpnEevqoZfyD.root',
    'password': 'oiK2dgnVVJLVHL4v',
    'database': 'data-mining',
    'charset': 'utf8mb4'
}

# ========== SSL CERTIFICATE ==========
CA_CERT_CONTENT = """
-----BEGIN CERTIFICATE-----
MIIFazCCA1OgAwIBAgIRAIIQz7DSQONZRGPgu2OCiwAwDQYJKoZIhvcNAQELBQAw
TzELMAkGA1UEBhMCVVMxKTAnBgNVBAoTIEludGVybmV0IFNlY3VyaXR5IFJlc2Vh
cmNoIEdyb3VwMRUwEwYDVQQDEwxJU1JHIFJvb3QgWDEwHhcNMTUwNjA0MTEwNDM4
WhcNMzUwNjA0MTEwNDM4WjBPMQswCQYDVQQGEwJVUzEpMCcGA1UEChMgSW50ZXJu
ZXQgU2VjdXJpdHkgUmVzZWFyY2ggR3JvdXAxFTATBgNVBAMTDElTUkcgUm9vdCBY
MTCCAiIwDQYJKoZIhvcNAQEBBQADggIPADCCAgoCggIBAK3oJHP0FDfzm54rVygc
h77ct984kIxuPOZXoHj3dcKi/vVqbvYATyjb3miGbESTtrFj/RQSa78f0uoxmyF+
0TM8ukj13Xnfs7j/EvEhmkvBioZxaUpmZmyPfjxwv60pIgbz5MDmgK7iS4+3mX6U
A5/TR5d8mUgjU+g4rk8Kb4Mu0UlXjIB0ttov0DiNewNwIRt18jA8+o+u3dpjq+sW
T8KOEUt+zwvo/7V3LvSye0rgTBIlDHCNAymg4VMk7BPZ7hm/ELNKjD+Jo2FR3qyH
B5T0Y3HsLuJvW5iB4YlcNHlsdu87kGJ55tukmi8mxdAQ4Q7e2RCOFvu396j3x+UC
B5iPNgiV5+I3lg02dZ77DnKxHZu8A/lJBdiB3QW0KtZB6awBdpUKD9jf1b0SHzUv
KBds0pjBqAlkd25HN7rOrFleaJ1/ctaJxQZBKT5ZPt0m9STJEadao0xAH0ahmbWn
OlFuhjuefXKnEgV4We0+UXgVCwOPjdAvBbI+e0ocS3MFEvzG6uBQE3xDk3SzynTn
jh8BCNAw1FtxNrQHusEwMFxIt4I7mKZ9YIqioymCzLq9gwQbooMDQaHWBfEbwrbw
qHyGO0aoSCqI3Haadr8faqU9GY/rOPNk3sgrDQoo//fb4hVC1CLQJ13hef4Y53CI
rU7m2Ys6xt0nUW7/vGT1M0NPAgMBAAGjQjBAMA4GA1UdDwEB/wQEAwIBBjAPBgNV
HRMBAf8EBTADAQH/MB0GA1UdDgQWBBR5tFnme7bl5AFzgAiIyBpY9umbbjANBgkq
hkiG9w0BAQsFAAOCAgEAVR9YqbyyqFDQDLHYGmkgJykIrGF1XIpu+ILlaS/V9lZL
ubhzEFnTIZd+50xx+7LSYK05qAvqFyFWhfFQDlnrzuBZ6brJFe+GnY+EgPbk6ZGQ
3BebYhtF8GaV0nxvwuo77x/Py9auJ/GpsMiu/X1+mvoiBOv/2X/qkSsisRcOj/KK
NFtY2PwByVS5uCbMiogziUwthDyC3+6WVwW6LLv3xLfHTjuCvjHIInNzktHCgKQ5
ORAzI4JMPJ+GslWYHb4phowim57iaztXOoJwTdwJx4nLCgdNbOhdjsnvzqvHu7Ur
TkXWStAmzOVyyghqpZXjFaH3pO3JLF+l+/+sKAIuvtd7u+Nxe5AW0wdeRlN8NwdC
jNPElpzVmbUq4JUagEiuTDkHzsxHpFKVK7q4+63SM1N95R1NbdWhscdCb+ZAJzVc
oyi3B43njTOQ5yOf+1CceWxG1bQVs5ZufpsMljq4Ui0/1lvh+wjChP4kqKOJ2qxq
4RgqsahDYVvTH9w7jXbyLeiNdd8XM2w9U/t7y0Ff/9yi0GE44Za4rF2LN9d11TPA
mRGunUHBcnWEvgJBQl9nJEiU0Zsnvgc/ubhPgXRR4Xq37Z0j4r7g1SgEEzwxA57d
emyPxgcYxn/eR44/KJ4EBs+lVDR3veyJm+kXQ99b21/+jh5Xos1AnX5iItreGCc=
-----END CERTIFICATE-----
""".strip()

CLIENT_CERT_CONTENT = None
CLIENT_KEY_CONTENT = None

print(f'📊 Database: {DB_CONFIG["database"]}')
print(f'🔗 Host: {DB_CONFIG["host"]}:{DB_CONFIG["port"]}')
print(f'📜 CA Cert: {"✅" if len(CA_CERT_CONTENT) > 100 else "❌ Not configured"}')

📊 Database: data-mining
🔗 Host: gateway01.ap-southeast-1.prod.aws.tidbcloud.com:4000
📜 CA Cert: ✅


## 4. Setup SSL Connection

In [4]:
temp_files = []
try:
    ssl_context = ssl.create_default_context()
    
    if CA_CERT_CONTENT:
        ca_temp = tempfile.NamedTemporaryFile(mode='w', suffix='.pem', delete=False)
        ca_temp.write(CA_CERT_CONTENT)
        ca_temp.close()
        temp_files.append(ca_temp.name)
        ssl_context.load_verify_locations(ca_temp.name)
        print('✅ Loaded CA certificate')
    
    if CLIENT_CERT_CONTENT and CLIENT_KEY_CONTENT:
        cert_temp = tempfile.NamedTemporaryFile(mode='w', suffix='.pem', delete=False)
        cert_temp.write(CLIENT_CERT_CONTENT)
        cert_temp.close()
        key_temp = tempfile.NamedTemporaryFile(mode='w', suffix='.pem', delete=False)
        key_temp.write(CLIENT_KEY_CONTENT)
        key_temp.close()
        temp_files.extend([cert_temp.name, key_temp.name])
        ssl_context.load_cert_chain(cert_temp.name, key_temp.name)
        print('✅ Loaded client certificate')
    
    DATABASE_URL = f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}?charset={DB_CONFIG['charset']}"
    engine = create_engine(DATABASE_URL, connect_args={'ssl': ssl_context}, pool_pre_ping=True)
    
    print('\n🔄 Testing connection...')
    with engine.connect() as conn:
        result = conn.execute(text('SELECT COUNT(*) FROM jobs'))
        total = result.fetchone()[0]
        print(f'\n✅ Connection successful!')
        print(f'📊 Total jobs: {total:,}')
        
except Exception as e:
    print(f'❌ Error: {e}')
    for f in temp_files:
        try: os.unlink(f)
        except: pass
    raise

✅ Loaded CA certificate

🔄 Testing connection...

✅ Connection successful!
📊 Total jobs: 12,117


## 5. Load Data from Database

In [5]:
print('📥 Loading jobs data...')
query = '''
SELECT 
    id, 
    title,
    description, 
    requirements_text, 
    required_skills,
    job_type,
    level,
    is_remote,
    location,
    company_name
FROM jobs 
ORDER BY id
'''

df = pd.read_sql(query, engine)
print(f'✅ Loaded {len(df):,} jobs')

# Show current state of target columns
print('\n📊 Current state of target columns:')
for col in ['job_type', 'level', 'is_remote']:
    null_count = df[col].isna().sum()
    filled_count = len(df) - null_count
    print(f'  {col}: {filled_count:,} filled, {null_count:,} empty ({null_count/len(df)*100:.1f}% empty)')

df.head()

📥 Loading jobs data...
✅ Loaded 12,117 jobs

📊 Current state of target columns:
  job_type: 12,117 filled, 0 empty (0.0% empty)
  level: 11,530 filled, 587 empty (4.8% empty)
  is_remote: 12,117 filled, 0 empty (0.0% empty)


,id,title,description,requirements_text,required_skills,job_type,level,is_remote,location,company_name
0,3462,Frontend developer (PA project),Your role & responsibilities:\nWe’re looking f...,Your skills & qualifications:\n\nExperience in...,"[""CSS"", ""HTML"", ""Git"", ""Restful Api"", ""SASS"", ...",Full-time,Manager,0,"Quận 1, Hồ Chí Minh",CUBICASA
1,3463,QA Engineer (Automotive),None,None,"[""QA"", ""Tester"", ""Unit Testing"", ""Integration ...",Full-time,None,0,"Thành phố Hồ Chí Minh, Hồ Chí Minh",42dot Vietnam
2,3464,Java Developer,None,None,"[""Java"", ""JavaScript"", ""JQuery"", ""VueJS"", ""Rea...",Full-time,None,0,"Quận Đống Đa, Hà Nội",WEEDS VINA
3,3465,Nhân viên chế bản - Webtoon/ Manga - Full-time,None,None,"[""Photoshop""]",Full-time,None,0,"Quận Bình Thạnh, Hồ Chí Minh",DaouKiwoom Innovation
4,3466,Webtoon Colorist - Họa sĩ tô màu - Full-time,None,None,None,Full-time,None,0,"Quận Bình Thạnh, Hồ Chí Minh",DaouKiwoom Innovation


## 6. Define Extraction Functions

Pattern-based extraction using keywords and regex

## 7. Check Current Inconsistencies

In [6]:
print('🔍 CHECKING CURRENT INCONSISTENCIES IN DATABASE\n')
print('='*100)

# Find Internship jobs with non-Intern levels
inconsistent_1 = df[(df['job_type'] == 'Internship') & (df['level'] != 'Intern') & df['level'].notna()]
print(f'\n❌ Internship jobs with wrong level: {len(inconsistent_1):,}')
if len(inconsistent_1) > 0:
    print('   Examples:')
    for idx in range(min(5, len(inconsistent_1))):
        job = inconsistent_1.iloc[idx]
        print(f'   • Job {job["id"]}: {job["title"]} | Type: {job["job_type"]} | Level: {job["level"]} ❌')

# Find Manager/Director/Lead with Internship type
inconsistent_2 = df[(df['level'].isin(['Manager', 'Director', 'Lead'])) & (df['job_type'] == 'Internship')]
print(f'\n❌ Manager/Director/Lead jobs marked as Internship: {len(inconsistent_2):,}')
if len(inconsistent_2) > 0:
    print('   Examples:')
    for idx in range(min(5, len(inconsistent_2))):
        job = inconsistent_2.iloc[idx]
        print(f'   • Job {job["id"]}: {job["title"]} | Type: {job["job_type"]} | Level: {job["level"]} ❌')

total_inconsistent = len(inconsistent_1) + len(inconsistent_2)
print(f'\n📊 Total inconsistencies found: {total_inconsistent:,}')
print(f'✅ These will be automatically fixed during extraction\n')
print('='*100)

🔍 CHECKING CURRENT INCONSISTENCIES IN DATABASE


❌ Internship jobs with wrong level: 0

❌ Manager/Director/Lead jobs marked as Internship: 0

📊 Total inconsistencies found: 0
✅ These will be automatically fixed during extraction



In [7]:
class JobMetadataExtractor:
    def __init__(self):
        # Job Type patterns (English only - comprehensive)
        self.job_type_patterns = {
            'Internship': [
                r'\bintern\b', r'internship', r'trainee', r'intern\s+(?:position|role)',
                r'student\s+(?:intern|position)', r'summer\s+intern'
            ],
            'Part-time': [
                r'part[\s-]?time', r'part\s+time', r'parttime',
                r'hourly', r'flexible\s+hours', r'few\s+hours'
            ],
            'Contract': [
                r'\bcontract\b', r'freelance', r'contractor', r'consulting',
                r'temporary', r'temp\s+(?:position|role)', r'project[\s-]?based',
                r'fixed[\s-]?term', r'short[\s-]?term'
            ],
            'Full-time': [
                r'full[\s-]?time', r'full\s+time', r'fulltime',
                r'permanent', r'regular', r'staff\s+(?:position|role)',
                r'direct\s+hire', r'long[\s-]?term'
            ]
        }
        
        # Level patterns (English only - comprehensive)
        self.level_patterns = {
            'Intern': [
                r'\bintern\b', r'internship', r'trainee'
            ],
            'Entry': [
                r'\bentry\b', r'entry[\s-]?level', r'fresher', r'graduate',
                r'beginner', r'no\s+experience', r'0\s+year'
            ],
            'Junior': [
                r'\bjunior\b', r'jr\.?[\s,]', r'junior\s+(?:developer|engineer|designer|analyst)',
                r'1\s+year', r'2\s+year', r'associate'
            ],
            'Mid': [
                r'\bmid\b', r'mid[\s-]?level', r'middle', r'intermediate',
                r'3\s+year', r'4\s+year', r'5\s+year'
            ],
            'Senior': [
                r'\bsenior\b', r'sr\.?[\s,]', r'senior\s+(?:developer|engineer|designer|analyst)',
                r'expert', r'principal', r'staff\s+(?:engineer|developer)',
                r'6\s+year', r'7\s+year', r'8\s+year', r'9\s+year'
            ],
            'Lead': [
                r'\blead\b', r'tech\s*lead', r'team\s*lead', r'technical\s+lead',
                r'squad\s+lead', r'leading', r'architect'
            ],
            'Manager': [
                r'\bmanager\b', r'management', r'head\s+of',
                r'project\s+manager', r'product\s+manager', r'engineering\s+manager'
            ],
            'Director': [
                r'\bdirector\b', r'\bvp\b', r'vice\s*president',
                r'chief', r'c-level', r'cto\b', r'ceo\b', r'cio\b', r'cpo\b', r'cfo\b'
            ]
        }
        
        # Remote patterns (English only - IMPROVED specificity)
        self.remote_patterns = {
            'strong': [
                # High confidence - explicit remote indicators
                r'\bremote\b',
                r'work\s+from\s+home',
                r'\bwfh\b',
                r'remote[\s-]?work',
                r'fully\s+remote',
                r'100%\s+remote',
                r'remote\s+position',
                r'remote\s+job',
                r'telecommute',
                r'work\s+remotely',
                r'distributed\s+team',
                r'anywhere\s+(?:in|from)',
                r'location[\s:]?\s*remote',
                r'remote[\s/]+hybrid'
            ],
            'hybrid': [
                # Hybrid indicators
                r'\bhybrid\b',
                r'hybrid\s+work',
                r'remote\s+and\s+office',
                r'office\s+and\s+remote',
                r'partly\s+remote',
                r'flexible\s+(?:location|workplace)',
                r'work\s+from\s+(?:home|office)',
                r'days?\s+(?:in|at)\s+office',
                r'days?\s+remote'
            ]
        }
        
        # On-site indicators (to avoid false positives)
        self.onsite_patterns = [
            r'on[\s-]?site\s+only',
            r'must\s+(?:be\s+)?(?:located|work|be)\s+(?:in|at)',
            r'office[\s-]?based',
            r'in[\s-]?person\s+only',
            r'no\s+remote',
            r'not\s+remote',
            r'relocate\s+to'
        ]
        
        self.stats = {
            'job_type_extracted': 0,
            'job_type_defaulted': 0,
            'level_extracted': 0,
            'level_from_title': 0,
            'level_from_years': 0,
            'level_from_description': 0,
            'level_from_requirements': 0,
            'remote_extracted': 0,
            'remote_from_location': 0,
            'remote_from_description': 0,
            'onsite_detected': 0,
            'no_data': 0,
            'fixed_inconsistencies': 0
        }
    
    def extract_years_of_experience(self, text):
        """Extract years of experience from text (English only)"""
        if not text or not str(text).strip():
            return None
        
        text = str(text).lower()
        
        # English-only patterns for years of experience
        patterns = [
            r'(\d+)\+?\s*(?:years?|yrs?)\s*(?:of\s*)?(?:experience|exp|working)',
            r'(?:at\s*least|minimum|min\.?|from)\s*(\d+)\s*(?:years?|yrs?)',
            r'(\d+)\s*(?:to|-|~)\s*(\d+)\s*(?:years?|yrs?)',
            r'experience[:\s]+(\d+)\+?\s*(?:years?|yrs?)',
            r'(\d+)\+\s*(?:years?|yrs?)',
            r'with\s+(\d+)\s*(?:years?|yrs?)',
            r'have\s+(\d+)\s*(?:years?|yrs?)',
            r'requires?\s+(\d+)\s*(?:years?|yrs?)',
            r'need\s+(\d+)\s*(?:years?|yrs?)',
            r'must\s+have\s+(\d+)\s*(?:years?|yrs?)'
        ]
        
        years_found = []
        for pattern in patterns:
            matches = re.finditer(pattern, text, re.IGNORECASE)
            for match in matches:
                try:
                    years = int(match.group(1))
                    years_found.append(years)
                except:
                    pass
        
        # Return the minimum (most conservative) years found
        return min(years_found) if years_found else None
    
    def map_years_to_level(self, years):
        """Map years of experience to job level"""
        if years is None:
            return None
        
        if years == 0:
            return 'Entry'
        elif years <= 1:
            return 'Junior'
        elif years <= 2:
            return 'Junior'
        elif years <= 4:
            return 'Mid'
        elif years <= 7:
            return 'Senior'
        elif years <= 10:
            return 'Lead'
        else:
            return 'Manager'
    
    def extract_job_type(self, text, level=None):
        """Extract job type from text with fallback logic"""
        if not text or not str(text).strip():
            # Default to Full-time if no data
            self.stats['job_type_defaulted'] += 1
            return 'Full-time'
        
        text = str(text).lower()
        
        # Check patterns in priority order (most specific first)
        priority_order = ['Internship', 'Contract', 'Part-time', 'Full-time']
        
        for job_type in priority_order:
            patterns = self.job_type_patterns[job_type]
            for pattern in patterns:
                if re.search(pattern, text, re.IGNORECASE):
                    return job_type
        
        # Fallback logic based on level
        if level == 'Intern':
            return 'Internship'
        
        # Default to Full-time (most common)
        self.stats['job_type_defaulted'] += 1
        return 'Full-time'
    
    def extract_level_from_text(self, text):
        """Extract level using keyword patterns"""
        if not text or not str(text).strip():
            return None
        
        text = str(text).lower()
        
        # Check patterns in priority order (most specific first)
        priority_order = ['Director', 'Manager', 'Lead', 'Senior', 'Mid', 'Junior', 'Entry', 'Intern']
        
        for level in priority_order:
            patterns = self.level_patterns[level]
            for pattern in patterns:
                if re.search(pattern, text, re.IGNORECASE):
                    return level
        
        return None
    
    def extract_level(self, title, description, requirements_text):
        """
        Extract job level with priority:
        1. Title (most reliable - e.g., "Senior Developer")
        2. Years from requirements_text (e.g., "5 years experience")
        3. Keywords in description
        4. Keywords in requirements_text
        """
        # Priority 1: Check title first (most accurate)
        if title and str(title).strip():
            level = self.extract_level_from_text(str(title))
            if level:
                self.stats['level_from_title'] += 1
                return level
        
        # Priority 2: Extract years from requirements_text
        if requirements_text and str(requirements_text).strip():
            years = self.extract_years_of_experience(str(requirements_text))
            if years is not None:
                level = self.map_years_to_level(years)
                if level:
                    self.stats['level_from_years'] += 1
                    return level
        
        # Priority 3: Check description for level keywords
        if description and str(description).strip():
            level = self.extract_level_from_text(str(description))
            if level:
                self.stats['level_from_description'] += 1
                return level
        
        # Priority 4: Check requirements_text for level keywords
        if requirements_text and str(requirements_text).strip():
            level = self.extract_level_from_text(str(requirements_text))
            if level:
                self.stats['level_from_requirements'] += 1
                return level
        
        return None
    
    def extract_is_remote(self, location, description, requirements_text):
        """
        Extract remote status with improved accuracy
        Returns: 1 (remote/hybrid), 0 (on-site), or None (uncertain)
        
        Priority:
        1. Location field (most reliable - "Remote", "Hybrid", specific cities)
        2. Strong remote patterns in description/requirements
        3. On-site patterns (negative indicators)
        """
        # Combine all text for analysis
        all_text = ' '.join([
            str(location) if location else '',
            str(description) if description else '',
            str(requirements_text) if requirements_text else ''
        ])
        
        if not all_text.strip():
            return None
        
        all_text_lower = all_text.lower()
        location_lower = str(location).lower() if location else ''
        
        # Priority 1: Check location field first (most reliable)
        if location_lower:
            # Strong remote indicators in location
            if re.search(r'\bremote\b', location_lower):
                self.stats['remote_from_location'] += 1
                return 1
            
            if re.search(r'\bhybrid\b', location_lower):
                self.stats['remote_from_location'] += 1
                return 1
            
            # Check if location mentions specific cities (indicates on-site)
            # Common city patterns in Vietnam, US, etc.
            city_patterns = [
                r'ho chi minh|hcm|saigon',
                r'ha noi|hanoi',
                r'da nang|danang',
                r'san francisco|sf',
                r'new york|nyc',
                r'los angeles|la',
                r'seattle',
                r'singapore',
                r'tokyo',
                r'london',
                r'sydney'
            ]
            
            for city_pattern in city_patterns:
                if re.search(city_pattern, location_lower):
                    # Has specific city = likely on-site
                    self.stats['onsite_detected'] += 1
                    return 0
        
        # Priority 2: Check for explicit on-site requirements
        for pattern in self.onsite_patterns:
            if re.search(pattern, all_text_lower, re.IGNORECASE):
                self.stats['onsite_detected'] += 1
                return 0
        
        # Priority 3: Check for strong remote indicators
        for pattern in self.remote_patterns['strong']:
            if re.search(pattern, all_text_lower, re.IGNORECASE):
                self.stats['remote_from_description'] += 1
                return 1
        
        # Priority 4: Check for hybrid indicators
        for pattern in self.remote_patterns['hybrid']:
            if re.search(pattern, all_text_lower, re.IGNORECASE):
                self.stats['remote_from_description'] += 1
                return 1
        
        # Default: If no clear indicators, assume on-site (most common)
        self.stats['onsite_detected'] += 1
        return 0
    
    def validate_and_fix_consistency(self, job_type, level):
        """Fix inconsistent job_type and level combinations"""
        if not job_type or not level:
            return job_type, level, False
        
        fixed = False
        
        # Rule 1: Internship should always be Intern level
        if job_type == 'Internship':
            if level != 'Intern':
                level = 'Intern'
                fixed = True
        
        # Rule 2: Intern level should be Internship job type
        if level == 'Intern':
            if job_type not in ['Internship', 'Full-time']:
                job_type = 'Internship'
                fixed = True
        
        # Rule 3: Manager/Director/Lead should not be Internship
        if level in ['Manager', 'Director', 'Lead']:
            if job_type == 'Internship':
                job_type = 'Full-time'
                fixed = True
        
        return job_type, level, fixed
    
    def extract_all(self, row):
        """Extract all metadata from a job row"""
        title = row.get('title', '')
        description = row.get('description', '')
        requirements_text = row.get('requirements_text', '')
        required_skills = row.get('required_skills', '')
        location = row.get('location', '')
        
        # Combine all text for job_type detection
        combined_text = ' '.join([
            str(title),
            str(description),
            str(requirements_text),
            str(required_skills)
        ])
        
        if not combined_text.strip():
            self.stats['no_data'] += 1
            return 'Full-time', None, None  # Default job_type even with no data
        
        # Extract level first (needed for job_type fallback)
        level = self.extract_level(title, description, requirements_text)
        
        # Extract job_type from all text (with level-based fallback)
        job_type = self.extract_job_type(combined_text, level)
        
        # Extract remote status (using location, description, requirements)
        is_remote = self.extract_is_remote(location, description, requirements_text)
        
        if job_type:
            self.stats['job_type_extracted'] += 1
        if level:
            self.stats['level_extracted'] += 1
        if is_remote is not None:
            self.stats['remote_extracted'] += 1
        
        return job_type, level, is_remote
    
    def print_stats(self):
        """Print extraction statistics"""
        print(f"📊 EXTRACTION STATS:")
        print(f"  job_type: {self.stats['job_type_extracted']:,}")
        print(f"    └─ defaulted to Full-time: {self.stats['job_type_defaulted']:,}")
        print(f"  level: {self.stats['level_extracted']:,}")
        print(f"    └─ from title: {self.stats['level_from_title']:,}")
        print(f"    └─ from years (requirements): {self.stats['level_from_years']:,}")
        print(f"    └─ from description: {self.stats['level_from_description']:,}")
        print(f"    └─ from requirements: {self.stats['level_from_requirements']:,}")
        print(f"  is_remote: {self.stats['remote_extracted']:,}")
        print(f"    └─ from location field: {self.stats['remote_from_location']:,}")
        print(f"    └─ from description/requirements: {self.stats['remote_from_description']:,}")
        print(f"    └─ on-site detected: {self.stats['onsite_detected']:,}")
        print(f"  fixed inconsistencies: {self.stats['fixed_inconsistencies']:,}")
        print(f"  no_data: {self.stats['no_data']:,}")

print('✅ JobMetadataExtractor class defined')
print('🔍 Extraction methods: job_type, level, is_remote')
print('🇬🇧 English patterns only (translated data)')
print()
print('📋 Job Type extraction:')
print('  🔍 Searches: Internship, Contract, Part-time, Full-time keywords')
print('  🎯 Fallback: If Intern level → Internship, else → Full-time (default)')
print('  ✅ GUARANTEES: ALL rows will have job_type filled')
print()
print('📋 Level extraction priority:')
print('  1️⃣  Title keywords (Senior, Junior, Manager, etc.)')
print('  2️⃣  Years from requirements_text (0=Entry, 1-2=Junior, 3-4=Mid, 5-7=Senior, 8-10=Lead, 11+=Manager)')
print('  3️⃣  Description keywords')
print('  4️⃣  Requirements keywords')
print()
print('📋 Remote extraction priority (IMPROVED):')
print('  1️⃣  Location field ("Remote", "Hybrid", or specific city names)')
print('  2️⃣  Explicit on-site requirements ("on-site only", "no remote")')
print('  3️⃣  Strong remote indicators ("remote", "work from home", "wfh", "fully remote")')
print('  4️⃣  Hybrid indicators ("hybrid", "flexible location")')
print('  5️⃣  Default: On-site (most common)')
print()
print('🔧 Auto-fixes inconsistent combinations (e.g., Internship + Manager → Internship + Intern)')


✅ JobMetadataExtractor class defined
🔍 Extraction methods: job_type, level, is_remote
🇬🇧 English patterns only (translated data)

📋 Job Type extraction:
  🔍 Searches: Internship, Contract, Part-time, Full-time keywords
  🎯 Fallback: If Intern level → Internship, else → Full-time (default)
  ✅ GUARANTEES: ALL rows will have job_type filled

📋 Level extraction priority:
  1️⃣  Title keywords (Senior, Junior, Manager, etc.)
  2️⃣  Years from requirements_text (0=Entry, 1-2=Junior, 3-4=Mid, 5-7=Senior, 8-10=Lead, 11+=Manager)
  3️⃣  Description keywords
  4️⃣  Requirements keywords

📋 Remote extraction priority (IMPROVED):
  1️⃣  Location field ("Remote", "Hybrid", or specific city names)
  2️⃣  Explicit on-site requirements ("on-site only", "no remote")
  3️⃣  Strong remote indicators ("remote", "work from home", "wfh", "fully remote")
  4️⃣  Hybrid indicators ("hybrid", "flexible location")
  5️⃣  Default: On-site (most common)

🔧 Auto-fixes inconsistent combinations (e.g., Internship + 

## 8. Extract Metadata from All Jobs

In [ ]:
import time

extractor = JobMetadataExtractor()
df_updated = df.copy()

print(f'🚀 Starting metadata extraction for {len(df):,} jobs')
print(f'🔥 Mode: FULL EXTRACTION - Will update ALL jobs (not just empty fields)')
print(f'🔧 Auto-fixing inconsistent combinations')
print(f'⏱️  Estimated time: ~{len(df)/1000:.1f} seconds\n')

start_time = time.time()
extracted_count = 0
fixed_count = 0

for idx in tqdm(range(len(df_updated)), desc='Extracting metadata'):
    row = df_updated.iloc[idx]
    
    # ALWAYS extract - fill all fields
    job_type, level, is_remote = extractor.extract_all(row)
    Merge branch 'develop' into staging
    # Validate and fix consistency
    original_job_type = job_type
    original_level = level
    job_type, level, was_fixed = extractor.validate_and_fix_consistency(job_type, level)
    
    if was_fixed:
        fixed_count += 1
        extractor.stats['fixed_inconsistencies'] += 1
    
    # Update ALL fields (override existing values)
    if job_type:
        df_updated.at[idx, 'job_type'] = job_type
        extracted_count += 1
    
    if level:
        df_updated.at[idx, 'level'] = level
        extracted_count += 1
    
    # Only update is_remote if empty (keep existing values for this field)
    if pd.isna(row['is_remote']) and is_remote is not None:
        df_updated.at[idx, 'is_remote'] = is_remote
        extracted_count += 1
    
    # Progress reporting every 500 jobs
    if (idx + 1) % 500 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed
        remaining = (len(df_updated) - idx - 1) / rate
        print(f'\n[{idx+1}/{len(df_updated)}] Time: {elapsed:.1f}s | '
              f'Remaining: {remaining:.1f}s | Rate: {rate:.1f} jobs/s')
        extractor.print_stats()

elapsed_time = time.time() - start_time
print(f'\n✅ Extraction complete!')
print(f'⏱️  Total time: {elapsed_time:.1f} seconds')
print(f'⚡ Average rate: {len(df)/elapsed_time:.1f} jobs/second')
print(f'\n📊 EXTRACTION RESULTS:')
extractor.print_stats()
print(f'🔧 Fixed inconsistencies: {fixed_count:,}')

# Show before/after comparison
print(f'\n📊 Before vs After:')
for col in ['job_type', 'level', 'is_remote']:
    before_null = df[col].isna().sum()
    after_null = df_updated[col].isna().sum()
    before_filled = len(df) - before_null
    after_filled = len(df) - after_null
    extracted = before_null - after_null
    print(f'  {col}: {before_filled:,} → {after_filled:,} filled '
          f'({extracted:,} new, {after_null:,} still empty)')

🚀 Starting metadata extraction for 12,117 jobs
🔥 Mode: FULL EXTRACTION - Will update ALL jobs (not just empty fields)
🔧 Auto-fixing inconsistent combinations
⏱️  Estimated time: ~12.1 seconds



Extracting metadata:   0%|          | 0/12117 [00:00<?, ?it/s]


[500/12117] Time: 2.4s | Remaining: 54.6s | Rate: 212.7 jobs/s
📊 EXTRACTION STATS:
  job_type: 500
    └─ defaulted to Full-time: 359
  level: 423
    └─ from title: 206
    └─ from years (requirements): 160
    └─ from description: 40
    └─ from requirements: 17
  is_remote: 500
    └─ from location field: 3
    └─ from description/requirements: 53
    └─ on-site detected: 444
  fixed inconsistencies: 4
  no_data: 0

[1000/12117] Time: 4.4s | Remaining: 49.1s | Rate: 226.3 jobs/s
📊 EXTRACTION STATS:
  job_type: 1,000
    └─ defaulted to Full-time: 672
  level: 913
    └─ from title: 435
    └─ from years (requirements): 344
    └─ from description: 117
    └─ from requirements: 17
  is_remote: 1,000
    └─ from location field: 3
    └─ from description/requirements: 150
    └─ on-site detected: 847
  fixed inconsistencies: 7
  no_data: 0

[1500/12117] Time: 5.5s | Remaining: 39.0s | Rate: 272.5 jobs/s
📊 EXTRACTION STATS:
  job_type: 1,500
    └─ defaulted to Full-time: 1,027
  level

## 9. Preview Results

In [9]:
print('🔍 SAMPLE EXTRACTIONS (First 10 jobs with changes)\n')
print('='*100)

shown = 0
for idx in range(len(df)):
    # Check if any field was updated or changed
    job_type_changed = str(df.at[idx, 'job_type']) != str(df_updated.at[idx, 'job_type'])
    level_changed = str(df.at[idx, 'level']) != str(df_updated.at[idx, 'level'])
    remote_changed = str(df.at[idx, 'is_remote']) != str(df_updated.at[idx, 'is_remote'])
    
    if job_type_changed or level_changed or remote_changed:
        job = df.iloc[idx]
        print(f'\n📋 Job ID: {job["id"]} | {job["title"]}')
        print(f'   Company: {job["company_name"]} | Location: {job["location"]}')
        
        if job_type_changed:
            old_val = df.at[idx, 'job_type'] if pd.notna(df.at[idx, 'job_type']) else 'NULL'
            new_val = df_updated.at[idx, 'job_type'] if pd.notna(df_updated.at[idx, 'job_type']) else 'NULL'
            print(f'   ✅ job_type: {old_val} → {new_val}')
        if level_changed:
            old_val = df.at[idx, 'level'] if pd.notna(df.at[idx, 'level']) else 'NULL'
            new_val = df_updated.at[idx, 'level'] if pd.notna(df_updated.at[idx, 'level']) else 'NULL'
            print(f'   ✅ level: {old_val} → {new_val}')
        if remote_changed:
            old_val = df.at[idx, 'is_remote'] if pd.notna(df.at[idx, 'is_remote']) else 'NULL'
            new_val = df_updated.at[idx, 'is_remote'] if pd.notna(df_updated.at[idx, 'is_remote']) else 'NULL'
            print(f'   ✅ is_remote: {old_val} → {new_val}')
        
        # Show excerpt from description
        if pd.notna(job['description']):
            desc_excerpt = str(job['description'])[:150]
            print(f'   📄 Description: {desc_excerpt}...')
        
        shown += 1
        if shown >= 10:
            break

print('\n' + '='*100)

# Show distribution of extracted values
print('\n📊 DISTRIBUTION OF EXTRACTED VALUES:\n')

print('job_type distribution:')
job_type_counts = df_updated['job_type'].value_counts()
for val, count in job_type_counts.items():
    print(f'  {val}: {count:,} ({count/len(df)*100:.1f}%)')

print('\nlevel distribution:')
level_counts = df_updated['level'].value_counts()
for val, count in level_counts.items():
    print(f'  {val}: {count:,} ({count/len(df)*100:.1f}%)')

print('\nis_remote distribution:')
remote_counts = df_updated['is_remote'].value_counts()
for val, count in remote_counts.items():
    label = 'Remote/Hybrid' if val == 1 else 'On-site'
    print(f'  {val} ({label}): {count:,} ({count/len(df)*100:.1f}%)')

🔍 SAMPLE EXTRACTIONS (First 10 jobs with changes)


📋 Job ID: 12683 | Junior Software Engineer
   Company: hackajob | Location: New York, NY
   ✅ job_type: Employment typeFull-time → Internship
   ✅ level: NULL → Intern
   📄 Description: hackajob
is collaborating with
mThree
to connect them with exceptional tech professionals for this role.
Locations:
New York City, NY
Salary:
$61,000 ...

📋 Job ID: 12684 | Software Engineer
   Company: Microsoft | Location: Redmond, WA
   ✅ job_type: Employment typeFull-time → Full-time
   ✅ level: NULL → Junior
   📄 Description: Overview
Team Purpose
The Microsoft Power Platform has embraced a shift to delivering agentic experiences that fuel customer productivity and vastly e...

📋 Job ID: 12685 | Software Development Engineer (Wallet, Payments & Commerce)
   Company: Apple | Location: Austin, TX
   ✅ job_type: Employment typeFull-time → Full-time
   ✅ level: Seniority levelEntry level → Lead
   📄 Description: Summary
Here at Apple, we build product

## 10. Save Backup CSV

In [10]:
from datetime import datetime

backup_file = f'jobs_metadata_extracted_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
df_updated.to_csv(backup_file, index=False, encoding='utf-8-sig')
print(f'✅ Backup saved: {backup_file}')
print(f'📦 File size: {os.path.getsize(backup_file) / 1024 / 1024:.2f} MB')

try:
    from google.colab import files
    files.download(backup_file)
    print('📥 File downloaded')
except:
    print('💾 File saved locally')

✅ Backup saved: jobs_metadata_extracted_20260121_115409.csv
📦 File size: 66.76 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 File downloaded


## 11. Update Database

In [11]:
confirm = input('⚠️  Update database with extracted metadata? (yes/no): ')

if confirm.lower() == 'yes':
    print('\n🔄 Updating database...')
    print('⏰ This will update the "updated_at" timestamp for all modified rows\n')
    
    update_count = 0
    error_count = 0
    
    with engine.begin() as conn:
        for idx in tqdm(range(len(df_updated)), desc='Updating database'):
            old_row = df.iloc[idx]
            new_row = df_updated.iloc[idx]
            
            # Only update if something changed
            job_type_changed = str(old_row['job_type']) != str(new_row['job_type'])
            level_changed = str(old_row['level']) != str(new_row['level'])
            remote_changed = str(old_row['is_remote']) != str(new_row['is_remote'])
            
            if job_type_changed or level_changed or remote_changed:
                try:
                    update_query = '''
                        UPDATE jobs SET
                            job_type = :job_type,
                            level = :level,
                            is_remote = :is_remote,
                            updated_at = NOW()
                        WHERE id = :id
                    '''
                    
                    params = {
                        'id': int(new_row['id']),
                        'job_type': new_row['job_type'] if pd.notna(new_row['job_type']) else None,
                        'level': new_row['level'] if pd.notna(new_row['level']) else None,
                        'is_remote': int(new_row['is_remote']) if pd.notna(new_row['is_remote']) else None
                    }
                    
                    conn.execute(text(update_query), params)
                    update_count += 1
                    
                except Exception as e:
                    error_count += 1
                    if error_count <= 5:
                        print(f'\n⚠️ Error updating job {new_row["id"]}: {str(e)[:100]}')
    
    print(f'\n✅ Updated: {update_count:,} jobs in database')
    print(f'❌ Errors: {error_count:,}')
    
    if error_count == 0:
        # Verify update
        print(f'\n🔍 Verifying database updates...')
        verify_query = 'SELECT COUNT(*) as updated FROM jobs WHERE updated_at >= DATE_SUB(NOW(), INTERVAL 5 MINUTE)'
        result = pd.read_sql(verify_query, engine)
        recently_updated = result['updated'].iloc[0]
        print(f'  ✅ {recently_updated:,} jobs have updated_at within last 5 minutes')
        
        # Show updated counts
        stats_query = '''
            SELECT 
                COUNT(*) as total,
                SUM(CASE WHEN job_type IS NOT NULL THEN 1 ELSE 0 END) as has_job_type,
                SUM(CASE WHEN level IS NOT NULL THEN 1 ELSE 0 END) as has_level,
                SUM(CASE WHEN is_remote IS NOT NULL THEN 1 ELSE 0 END) as has_remote
            FROM jobs
        '''
        stats = pd.read_sql(stats_query, engine).iloc[0]
        
        print(f'\n📊 Database statistics:')
        print(f'  Total jobs: {stats["total"]:,}')
        print(f'  Has job_type: {stats["has_job_type"]:,} ({stats["has_job_type"]/stats["total"]*100:.1f}%)')
        print(f'  Has level: {stats["has_level"]:,} ({stats["has_level"]/stats["total"]*100:.1f}%)')
        print(f'  Has is_remote: {stats["has_remote"]:,} ({stats["has_remote"]/stats["total"]*100:.1f}%)')
        
        print(f'\n✅ SUCCESS: Database updated successfully!')
    else:
        print(f'\n⚠️ WARNING: {error_count} errors occurred during update')
else:
    print('❌ Update cancelled by user')


🔄 Updating database...
⏰ This will update the "updated_at" timestamp for all modified rows



Updating database:   0%|          | 0/12117 [00:00<?, ?it/s]


✅ Updated: 2,896 jobs in database
❌ Errors: 0

🔍 Verifying database updates...
  ✅ 1,806 jobs have updated_at within last 5 minutes

📊 Database statistics:
  Total jobs: 12,117.0
  Has job_type: 12,117.0 (100.0%)
  Has level: 11,862.0 (97.9%)
  Has is_remote: 12,117.0 (100.0%)

✅ SUCCESS: Database updated successfully!


## 12. Cleanup and Summary

In [12]:
# Cleanup temporary SSL files
for f in temp_files:
    try:
        os.unlink(f)
        print(f'🗑️  Deleted temp file: {f}')
    except:
        pass

print('\n' + '='*80)
print('🎉 METADATA EXTRACTION COMPLETE - FINAL SUMMARY')
print('='*80)
print(f'📊 Total jobs processed: {len(df):,}')
print(f'💾 Backup file: {backup_file}')
print(f'⏱️  Processing time: {elapsed_time:.1f} seconds')
print()

print('📈 EXTRACTION STATISTICS:')
extractor.print_stats()
print()

print('📊 FIELD COVERAGE (after extraction):')
for col in ['job_type', 'level', 'is_remote']:
    filled = df_updated[col].notna().sum()
    print(f'  {col}: {filled:,} / {len(df):,} ({filled/len(df)*100:.1f}%)')
print()

print('='*80)

🗑️  Deleted temp file: /tmp/tmp2yhw93vp.pem

🎉 METADATA EXTRACTION COMPLETE - FINAL SUMMARY
📊 Total jobs processed: 12,117
💾 Backup file: jobs_metadata_extracted_20260121_115409.csv
⏱️  Processing time: 61.9 seconds

📈 EXTRACTION STATISTICS:
📊 EXTRACTION STATS:
  job_type: 12,117
    └─ defaulted to Full-time: 6,881
  level: 11,416
    └─ from title: 4,275
    └─ from years (requirements): 3,447
    └─ from description: 3,493
    └─ from requirements: 201
  is_remote: 12,117
    └─ from location field: 3
    └─ from description/requirements: 2,293
    └─ on-site detected: 9,821
  fixed inconsistencies: 238
  no_data: 0

📊 FIELD COVERAGE (after extraction):
  job_type: 12,117 / 12,117 (100.0%)
  level: 11,862 / 12,117 (97.9%)
  is_remote: 12,117 / 12,117 (100.0%)

